In [8]:
import pandas as pd
import os
import altair as alt
from vega_datasets import data as vega_data

df = pd.read_csv(os.path.expanduser("~/Desktop/DS4200/eia_data_with_derived_variables.csv"))
df.head()

,State,Year,Coal_Consumption,CO2_Emissions,NatGas_Consumption,Nuclear_Consumption,Renewable_Consumption,Total_Energy_Consumption,Region,Dominant_Energy_Source
0,AK,1997,11719.0,589.0,425394.0,0.0,8109.0,701267.0,West,Natural Gas
1,AK,1998,16471.0,347.0,434437.0,0.0,6084.0,715652.0,West,Natural Gas
2,AK,1999,16394.0,392.0,422817.0,0.0,5040.0,719766.0,West,Natural Gas
3,AK,2000,16455.0,171.0,437972.0,0.0,5574.0,735260.0,West,Natural Gas
4,AK,2001,15911.0,465.0,413049.0,0.0,8119.0,726418.0,West,Natural Gas


In [9]:
# Average CO2 per state across all years
state_avg = (
    df.groupby("State", as_index=False)
    .agg(
        Avg_CO2=("CO2_Emissions", "mean"),
        Avg_Total_Energy=("Total_Energy_Consumption", "mean"),
        Dominant_Energy_Source=("Dominant_Energy_Source", lambda x: x.mode()[0]),
        Region=("Region", "first"),
    )
)

# State FIPS lookup
fips_lookup = {
    "AL":1,"AK":2,"AZ":4,"AR":5,"CA":6,"CO":8,"CT":9,"DE":10,"FL":12,"GA":13,
    "HI":15,"ID":16,"IL":17,"IN":18,"IA":19,"KS":20,"KY":21,"LA":22,"ME":23,
    "MD":24,"MA":25,"MI":26,"MN":27,"MS":28,"MO":29,"MT":30,"NE":31,"NV":32,
    "NH":33,"NJ":34,"NM":35,"NY":36,"NC":37,"ND":38,"OH":39,"OK":40,"OR":41,
    "PA":42,"RI":44,"SC":45,"SD":46,"TN":47,"TX":48,"UT":49,"VT":50,"VA":51,
    "WA":53,"WV":54,"WI":55,"WY":56,"DC":11,
}
state_avg["id"] = state_avg["State"].map(fips_lookup)
state_avg = state_avg.dropna(subset=["id"])
state_avg["id"] = state_avg["id"].astype(int)

states_topo = alt.topo_feature(
    "https://cdn.jsdelivr.net/npm/vega-datasets@1/data/us-10m.json",
    "states"
)

# Visualization 1: Choropleth Map — Average CO₂ Emissions by State
choropleth = (
    alt.Chart(states_topo)
    .mark_geoshape(stroke="white", strokeWidth=0.5)
    .transform_lookup(
        lookup="id",
        from_=alt.LookupData(state_avg, "id", [
            "State", "Avg_CO2", "Avg_Total_Energy",
            "Dominant_Energy_Source", "Region"
        ])
    )
    .encode(
        color=alt.Color(
            "Avg_CO2:Q",
            scale=alt.Scale(scheme="orangered"),
            title="Avg CO₂ Emissions",
        ),
        tooltip=[
            alt.Tooltip("State:N",                  title="State"),
            alt.Tooltip("Region:N",                 title="Region"),
            alt.Tooltip("Avg_CO2:Q",                title="Avg CO₂ (metric tons)", format=",.0f"),
            alt.Tooltip("Avg_Total_Energy:Q",       title="Avg Total Energy",      format=",.0f"),
            alt.Tooltip("Dominant_Energy_Source:N", title="Dominant Source"),
        ],
    )
    .project("albersUsa")
    .properties(
        width=650,
        height=400,
        title="Average CO₂ Emissions by State (all years)",
    )
)

choropleth

/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/c

alt.Chart(...)

In [ ]:
# Visualization 2: Boxplot — CO₂ Emissions by Region and Energy Source
df_box = df.dropna(subset=["Region", "Dominant_Energy_Source", "CO2_Emissions"]).copy()

df_box["Q1"]     = df_box.groupby(["Region","Dominant_Energy_Source"])["CO2_Emissions"].transform(lambda x: x.quantile(0.25))
df_box["Median"] = df_box.groupby(["Region","Dominant_Energy_Source"])["CO2_Emissions"].transform("median")
df_box["Q3"]     = df_box.groupby(["Region","Dominant_Energy_Source"])["CO2_Emissions"].transform(lambda x: x.quantile(0.75))
df_box["IQR"]    = df_box["Q3"] - df_box["Q1"]
df_box["Min"]    = df_box.groupby(["Region","Dominant_Energy_Source"])["CO2_Emissions"].transform("min")
df_box["Max"]    = df_box.groupby(["Region","Dominant_Energy_Source"])["CO2_Emissions"].transform("max")
df_box["Whisker_Low"]  = df_box["Q1"] - 1.5 * df_box["IQR"]
df_box["Whisker_High"] = df_box["Q3"] + 1.5 * df_box["IQR"]

base = alt.Chart(df_box)
boxes = base.mark_boxplot(
    extent=1.5,
    size=40,
    outliers=alt.MarkConfig(size=12, opacity=0.3),
).encode(
    x=alt.X("Region:N", title=None, axis=alt.Axis(labelAngle=-20)),
    y=alt.Y("CO2_Emissions:Q", title="CO₂ Emissions (metric tons)"),
    color=alt.Color("Region:N", title="Region",
                    legend=alt.Legend(orient="right")),
)

tooltip_layer = base.mark_point(opacity=0, size=900).encode(
    x=alt.X("Region:N"),
    y=alt.Y("Median:Q"),
    color=alt.Color("Region:N", legend=None),
    tooltip=[
        alt.Tooltip("Dominant_Energy_Source:N", title="Energy Source"),
        alt.Tooltip("Region:N",                 title="Region"),
        alt.Tooltip("Median:Q", title="Median CO₂",  format=",.0f"),
        alt.Tooltip("Q1:Q",     title="Q1 (25th %)", format=",.0f"),
        alt.Tooltip("Q3:Q",     title="Q3 (75th %)", format=",.0f"),
        alt.Tooltip("IQR:Q",    title="IQR",         format=",.0f"),
        alt.Tooltip("Min:Q",    title="Min CO₂",     format=",.0f"),
        alt.Tooltip("Max:Q",    title="Max CO₂",     format=",.0f"),
    ],
)

boxplot = (
    alt.layer(boxes, tooltip_layer)
    .facet(column=alt.Column(
        "Dominant_Energy_Source:N",
        title="Dominant Energy Source",
        header=alt.Header(labelFontSize=12)
    ))
    .properties(title="CO₂ Emissions by Energy Source and Region")
    .resolve_scale(y="shared")
)

boxplot

/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/c

alt.FacetChart(...)

In [10]:
choropleth.save("choropleth.json")
boxplot.save("boxplot.json")

/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/core.py:395: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  col = df[col_name].apply(to_list_if_array, convert_dtype=False)
/opt/anaconda3/lib/python3.12/site-packages/altair/utils/c